# Imports & Configuration

In [ ]:
import os
import math
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.parallel import DataParallel

from sklearn.preprocessing import StandardScaler
from sklearn import metrics as sk_metrics
from scipy.spatial import cKDTree

# ── CONFIG ────────────────────────────────────────────────────────────────────
CONFIG = {
    # Data paths
    "pressure_csv": "/kaggle/input/master-2sec-updated/master_pressure_2sec.csv",
    "u_velocity_csv": "/kaggle/input/master-2sec-updated/master_u_velocity_2sec.csv",
    "v_velocity_csv": "/kaggle/input/master-2sec-updated/master_v_velocity_2sec.csv",
    "chunk_size": None,
    "timestep_duration": 0.02,

    # Spatial
    "k_neighbors": 20,

    # Temporal
    "input_sequence_length": 30,
    "prediction_horizon_steps": 15,

    # Model
    "model_type": "SpatioTemporalLSTM",  # or "SpatioTemporalTransformer"
    "spatial_embed_dim": 64,
    "hidden_size": 256,
    "num_layers": 3,
    "nhead": 8,
    "dropout": 0.2,

    # Training
    "batch_size": 128,
    "learning_rate": 5e-4,
    "epochs": 80,
    "patience": 10,
    "grad_clip": 0.5,
    "loss_weights": {"pressure": 0.3, "u_velocity": 0.35, "v_velocity": 0.35},

    # Split ratios
    "train_ratio": 0.70,
    "val_ratio": 0.15,

    # Output
    "output_dir": "outputs",
    "save_predictions": True,
    "create_visualizations": True,
    "visualization_samples": [0, 5, 10],
}

# ── Derived values ────────────────────────────────────────────────────────────
PRED_STEPS = CONFIG["prediction_horizon_steps"]
IS_PINN = False

# ── Device setup ──────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")

os.makedirs(CONFIG["output_dir"], exist_ok=True)

# Data Loading

In [ ]:
def parse_urans_multifile(pressure_path, u_path, v_path, chunk_size=None):
    """
    Load three URANS CSV files (pressure, u-velocity, v-velocity).

    Each CSV has:
      - First 2 columns : x, y coordinates
      - Remaining columns: variable values at each timestep

    Returns
    -------
    coords          : np.ndarray, shape (n_cells, 2)
    data            : np.ndarray, shape (n_cells, n_timesteps, 3)  — order [P, U, V]
    n_timesteps     : int
    dataframes_dict : dict with keys 'pressure', 'u_velocity', 'v_velocity'
    """
    print("Loading CSV files ...")
    df_p = pd.read_csv(pressure_path)
    df_u = pd.read_csv(u_path)
    df_v = pd.read_csv(v_path)

    # Validate row counts
    assert len(df_p) == len(df_u) == len(df_v), (
        f"Row count mismatch: P={len(df_p)}, U={len(df_u)}, V={len(df_v)}"
    )
    n_cells = len(df_p)
    print(f"  n_cells = {n_cells:,}")

    # Extract coordinates (first 2 columns)
    coords_p = df_p.iloc[:, :2].values.astype(np.float64)
    coords_u = df_u.iloc[:, :2].values.astype(np.float64)
    coords_v = df_v.iloc[:, :2].values.astype(np.float64)

    assert np.allclose(coords_p, coords_u, atol=1e-8) and np.allclose(coords_p, coords_v, atol=1e-8), \
        "Coordinate mismatch across CSV files!"
    coords = coords_p  # shape (n_cells, 2)

    # Extract time-series data (columns 2+)
    ts_p = df_p.iloc[:, 2:].values.astype(np.float32)  # (n_cells, n_timesteps)
    ts_u = df_u.iloc[:, 2:].values.astype(np.float32)
    ts_v = df_v.iloc[:, 2:].values.astype(np.float32)

    assert ts_p.shape[1] == ts_u.shape[1] == ts_v.shape[1], (
        f"Timestep count mismatch: P={ts_p.shape[1]}, U={ts_u.shape[1]}, V={ts_v.shape[1]}"
    )
    n_timesteps = ts_p.shape[1]
    print(f"  n_timesteps = {n_timesteps}")

    # Apply chunk_size if requested
    if chunk_size is not None:
        n_cells = min(n_cells, chunk_size)
        coords = coords[:n_cells]
        ts_p = ts_p[:n_cells]
        ts_u = ts_u[:n_cells]
        ts_v = ts_v[:n_cells]
        print(f"  Using chunk_size={chunk_size}, effective n_cells={n_cells:,}")

    # Stack into (n_cells, n_timesteps, 3) — order [P, U, V]
    data = np.stack([ts_p, ts_u, ts_v], axis=-1)  # (n_cells, n_timesteps, 3)
    print(f"  data shape: {data.shape}")

    dataframes_dict = {
        "pressure": df_p,
        "u_velocity": df_u,
        "v_velocity": df_v,
    }
    return coords, data, n_timesteps, dataframes_dict

# Spatial Neighborhood Construction

In [ ]:
def build_knn_graph(coords, k):
    """
    For each cell, find its k nearest spatial neighbors (excluding self).

    Parameters
    ----------
    coords : np.ndarray, shape (n_cells, 2)
    k      : int — number of neighbors

    Returns
    -------
    neighbor_indices : np.ndarray, shape (n_cells, k)
    """
    print(f"Building kNN graph with k={k} ...")
    tree = cKDTree(coords)
    # Query k+1 because the first result is the point itself
    _, indices = tree.query(coords, k=k + 1)
    neighbor_indices = indices[:, 1:]  # exclude self → (n_cells, k)
    print(f"  neighbor_indices shape: {neighbor_indices.shape}")
    return neighbor_indices

# Per-Variable Normalization

In [ ]:
def normalize_per_variable(data, train_end_t):
    """
    Normalize each variable (P, U, V) independently using StandardScaler.
    Scalers are fit only on the training time range to prevent data leakage.

    Parameters
    ----------
    data        : np.ndarray, shape (n_cells, n_timesteps, 3)
    train_end_t : int — exclusive end index of the training time range

    Returns
    -------
    data_normalized : np.ndarray, shape (n_cells, n_timesteps, 3)
    scalers         : list of 3 StandardScaler instances [scaler_p, scaler_u, scaler_v]
    """
    print("Normalizing per variable (fit on training split only) ...")
    n_cells, n_timesteps, n_vars = data.shape
    data_normalized = np.empty_like(data)
    scalers = []

    var_names = ["Pressure", "U-velocity", "V-velocity"]
    for var_idx in range(n_vars):
        scaler = StandardScaler()
        # Fit only on training timesteps: flatten (n_cells, train_end_t) → 1-D
        train_slice = data[:, :train_end_t, var_idx].reshape(-1, 1)
        scaler.fit(train_slice)
        # Transform all timesteps
        all_slice = data[:, :, var_idx].reshape(-1, 1)
        data_normalized[:, :, var_idx] = scaler.transform(all_slice).reshape(n_cells, n_timesteps)
        scalers.append(scaler)
        print(f"  {var_names[var_idx]}: mean={scaler.mean_[0]:.4f}, std={scaler.scale_[0]:.4f}")

    return data_normalized, scalers

# Dataset

In [ ]:
class SpatioTemporalDataset(Dataset):
    """
    Sliding-window dataset for spatiotemporal flow forecasting.

    Each sample contains:
      - center_seq   : (seq_len, 3)     — the target cell's P/U/V history
      - neighbor_seq : (seq_len, k, 3)  — its k neighbors' P/U/V history
      - target       : (pred_steps, 3)  — ground truth future values
    """

    def __init__(self, data_normalized, neighbor_indices, seq_length, pred_steps, start_t, end_t):
        """
        Parameters
        ----------
        data_normalized  : np.ndarray, shape (n_cells, n_timesteps, 3)
        neighbor_indices : np.ndarray, shape (n_cells, k)
        seq_length       : int
        pred_steps       : int
        start_t          : int — inclusive start of valid time range
        end_t            : int — exclusive end of valid time range
        """
        super().__init__()
        self.data = torch.FloatTensor(data_normalized)
        self.neighbor_indices = torch.LongTensor(neighbor_indices)
        self.seq_length = seq_length
        self.pred_steps = pred_steps
        self.start_t = start_t

        self.n_cells = data_normalized.shape[0]
        # Number of valid window starting positions within [start_t, end_t)
        self.n_valid_windows = max(0, end_t - start_t - seq_length - pred_steps + 1)

    def __len__(self):
        return self.n_cells * self.n_valid_windows

    def __getitem__(self, idx):
        cell_idx = idx // self.n_valid_windows
        window_idx = idx % self.n_valid_windows
        t_start = self.start_t + window_idx

        # Center cell sequence: (seq_len, 3)
        center_seq = self.data[cell_idx, t_start:t_start + self.seq_length, :]

        # Neighbor sequences: gather (k, seq_len, 3), then permute → (seq_len, k, 3)
        nbr_indices = self.neighbor_indices[cell_idx]  # (k,)
        neighbor_seq = self.data[nbr_indices, t_start:t_start + self.seq_length, :]  # (k, seq_len, 3)
        neighbor_seq = neighbor_seq.permute(1, 0, 2)  # (seq_len, k, 3)

        # Target: (pred_steps, 3)
        target = self.data[cell_idx, t_start + self.seq_length:t_start + self.seq_length + self.pred_steps, :]

        return center_seq, neighbor_seq, target

# Model Architectures

In [ ]:
class SpatialEncoder(nn.Module):
    """
    Encodes the k-nearest-neighbor flow variables into a spatial embedding
    via a shared MLP followed by mean-pooling over the neighbor dimension.
    """

    def __init__(self, input_features=3, embed_dim=64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_features, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def forward(self, neighbor_seq):
        # neighbor_seq: (B, seq_len, k, 3)
        embedded = self.mlp(neighbor_seq)   # (B, seq_len, k, embed_dim)
        pooled = embedded.mean(dim=2)        # (B, seq_len, embed_dim)
        return pooled


class SpatioTemporalLSTM(nn.Module):
    """
    Spatiotemporal LSTM:
      1. SpatialEncoder aggregates neighbor features → spatial embedding
      2. Concatenate center-cell features with spatial embedding
      3. LSTM processes the combined sequence
      4. Decoder MLP maps last hidden state → (pred_steps, 3)
    """

    def __init__(self, input_features=3, spatial_embed_dim=64, hidden_size=256,
                 num_layers=3, pred_steps=15, dropout=0.2):
        super().__init__()
        self.pred_steps = pred_steps
        self.input_features = input_features
        self.spatial_encoder = SpatialEncoder(input_features, spatial_embed_dim)
        combined_dim = input_features + spatial_embed_dim
        self.lstm = nn.LSTM(
            combined_dim, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,  # PyTorch LSTM dropout is only applied between layers (not on single-layer)
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, pred_steps * input_features),
        )

    def forward(self, center_seq, neighbor_seq):
        # center_seq   : (B, seq_len, 3)
        # neighbor_seq : (B, seq_len, k, 3)
        spatial_embed = self.spatial_encoder(neighbor_seq)          # (B, seq_len, spatial_embed_dim)
        combined = torch.cat([center_seq, spatial_embed], dim=-1)   # (B, seq_len, 3+spatial_embed_dim)
        lstm_out, _ = self.lstm(combined)
        last_hidden = lstm_out[:, -1, :]                            # (B, hidden_size)
        output = self.decoder(last_hidden)                          # (B, pred_steps * 3)
        return output.view(-1, self.pred_steps, self.input_features)


class SpatioTemporalTransformer(nn.Module):
    """
    Spatiotemporal Transformer:
      1. SpatialEncoder aggregates neighbor features → spatial embedding
      2. Concatenate center-cell features with spatial embedding
      3. Linear projection to d_model + learned positional encoding
      4. Transformer encoder processes the sequence
      5. Decoder MLP maps last-position output → (pred_steps, 3)
    """

    def __init__(self, input_features=3, spatial_embed_dim=64, d_model=256,
                 nhead=8, num_layers=3, pred_steps=15, dropout=0.2):
        super().__init__()
        self.pred_steps = pred_steps
        self.input_features = input_features
        self.spatial_encoder = SpatialEncoder(input_features, spatial_embed_dim)
        combined_dim = input_features + spatial_embed_dim
        self.input_projection = nn.Linear(combined_dim, d_model)
        # Learned positional encoding (max 512 positions)
        self.pos_encoding = nn.Parameter(torch.randn(1, 512, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4, dropout=dropout, batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.decoder = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, pred_steps * input_features),
        )

    def forward(self, center_seq, neighbor_seq):
        # center_seq   : (B, seq_len, 3)
        # neighbor_seq : (B, seq_len, k, 3)
        spatial_embed = self.spatial_encoder(neighbor_seq)          # (B, seq_len, spatial_embed_dim)
        combined = torch.cat([center_seq, spatial_embed], dim=-1)   # (B, seq_len, 3+spatial_embed_dim)
        x = self.input_projection(combined)                         # (B, seq_len, d_model)
        seq_len = x.shape[1]
        x = x + self.pos_encoding[:, :seq_len, :]                   # add positional encoding
        x = self.transformer_encoder(x)                             # (B, seq_len, d_model)
        last_output = x[:, -1, :]                                   # (B, d_model)
        output = self.decoder(last_output)                          # (B, pred_steps * 3)
        return output.view(-1, self.pred_steps, self.input_features)


def create_model(cfg):
    """
    Factory function that creates a model based on cfg['model_type'].

    Parameters
    ----------
    cfg : dict — CONFIG dictionary

    Returns
    -------
    model : nn.Module
    """
    model_type = cfg["model_type"]
    pred_steps = cfg["prediction_horizon_steps"]
    spatial_embed_dim = cfg["spatial_embed_dim"]
    hidden_size = cfg["hidden_size"]
    num_layers = cfg["num_layers"]
    dropout = cfg["dropout"]

    if model_type == "SpatioTemporalLSTM":
        model = SpatioTemporalLSTM(
            input_features=3,
            spatial_embed_dim=spatial_embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            pred_steps=pred_steps,
            dropout=dropout,
        )
    elif model_type == "SpatioTemporalTransformer":
        model = SpatioTemporalTransformer(
            input_features=3,
            spatial_embed_dim=spatial_embed_dim,
            d_model=hidden_size,
            nhead=cfg["nhead"],
            num_layers=num_layers,
            pred_steps=pred_steps,
            dropout=dropout,
        )
    else:
        raise ValueError(f"Unknown model_type: '{model_type}'. "
                         "Choose 'SpatioTemporalLSTM' or 'SpatioTemporalTransformer'.")

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Created {model_type} with {total_params:,} trainable parameters.")
    return model

# Loss Function

In [ ]:
class WeightedMSELoss(nn.Module):
    """
    Weighted MSE loss that applies per-channel weights to the squared error.

    Channel order is [P, U, V] matching the data array.
    Default weights: P=0.3, U=0.35, V=0.35.
    """

    def __init__(self, weights):
        """
        Parameters
        ----------
        weights : dict with keys 'pressure', 'u_velocity', 'v_velocity'
        """
        super().__init__()
        w = torch.tensor(
            [weights["pressure"], weights["u_velocity"], weights["v_velocity"]],
            dtype=torch.float32,
        ).view(1, 1, 3)  # broadcast over (batch, pred_steps, channels)
        self.register_buffer("weights", w)

    def forward(self, pred, target):
        """
        Parameters
        ----------
        pred   : Tensor, shape (B, pred_steps, 3)
        target : Tensor, shape (B, pred_steps, 3)
        """
        return ((pred - target) ** 2 * self.weights).mean()

# Training & Evaluation

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    """
    Run one training epoch.

    Returns
    -------
    dict with key 'loss' (float)
    """
    model.train()
    total_loss = 0.0
    n_batches = 0

    for center_seq, neighbor_seq, target in tqdm(dataloader, desc="  Train", leave=False):
        center_seq = center_seq.to(device, non_blocking=True)
        neighbor_seq = neighbor_seq.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)

        optimizer.zero_grad()
        pred = model(center_seq, neighbor_seq)
        loss = criterion(pred, target)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return {"loss": total_loss / max(n_batches, 1)}


def evaluate(model, dataloader, criterion, scalers, device):
    """
    Evaluate the model on a dataloader.

    Returns
    -------
    dict with keys:
      loss, rmse, mae,
      pressure_rmse, u_rmse, v_rmse,
      predictions (np.ndarray), targets (np.ndarray)
    """
    model.eval()
    total_loss = 0.0
    n_batches = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for center_seq, neighbor_seq, target in tqdm(dataloader, desc="  Eval", leave=False):
            center_seq = center_seq.to(device, non_blocking=True)
            neighbor_seq = neighbor_seq.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)

            pred = model(center_seq, neighbor_seq)
            loss = criterion(pred, target)
            total_loss += loss.item()
            n_batches += 1

            all_preds.append(pred.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    all_preds = np.concatenate(all_preds, axis=0)    # (N, pred_steps, 3)
    all_targets = np.concatenate(all_targets, axis=0)

    avg_loss = total_loss / max(n_batches, 1)

    # Overall RMSE / MAE on normalized values
    rmse = float(np.sqrt(np.mean((all_preds - all_targets) ** 2)))
    mae = float(np.mean(np.abs(all_preds - all_targets)))

    # Per-variable RMSE on denormalized values
    var_names = ["pressure", "u", "v"]
    var_rmse = {}
    for vi, (name, scaler) in enumerate(zip(var_names, scalers)):
        pred_v = all_preds[:, :, vi].reshape(-1, 1)
        tgt_v = all_targets[:, :, vi].reshape(-1, 1)
        pred_denorm = scaler.inverse_transform(pred_v).ravel()
        tgt_denorm = scaler.inverse_transform(tgt_v).ravel()
        var_rmse[name] = float(np.sqrt(np.mean((pred_denorm - tgt_denorm) ** 2)))

    return {
        "loss": avg_loss,
        "rmse": rmse,
        "mae": mae,
        "pressure_rmse": var_rmse["pressure"],
        "u_rmse": var_rmse["u"],
        "v_rmse": var_rmse["v"],
        "predictions": all_preds,
        "targets": all_targets,
    }

# Export Predictions

In [ ]:
def export_predictions_to_csv(model, data_normalized, neighbor_indices, scalers, cfg, dataframes_original, device):
    """
    For each cell, use the last seq_length timesteps as input, predict the next
    pred_steps timesteps, denormalize, and save 3 CSVs with prediction columns
    appended to the original coordinates.

    Saves:
      outputs/pred_pressure.csv
      outputs/pred_u_velocity.csv
      outputs/pred_v_velocity.csv
    """
    print("Exporting predictions to CSV ...")
    model.eval()
    seq_len = cfg["input_sequence_length"]
    pred_steps = cfg["prediction_horizon_steps"]
    batch_size = cfg["batch_size"]
    n_cells = data_normalized.shape[0]
    n_timesteps = data_normalized.shape[1]

    data_tensor = torch.FloatTensor(data_normalized)
    nbr_tensor = torch.LongTensor(neighbor_indices)

    all_preds = np.zeros((n_cells, pred_steps, 3), dtype=np.float32)

    with torch.no_grad():
        for start in tqdm(range(0, n_cells, batch_size), desc="  Exporting"):
            end = min(start + batch_size, n_cells)
            # Last seq_len timesteps as input
            center_seq = data_tensor[start:end, n_timesteps - seq_len:, :]  # (bs, seq_len, 3)
            nbr_idx = nbr_tensor[start:end]                                  # (bs, k)
            neighbor_seq = data_tensor[nbr_idx, :, :]                       # (bs, k, n_timesteps, 3)
            # Slice last seq_len: (bs, k, seq_len, 3) → permute → (bs, seq_len, k, 3)
            neighbor_seq = neighbor_seq[:, :, n_timesteps - seq_len:, :].permute(0, 2, 1, 3)

            center_seq = center_seq.to(device)
            neighbor_seq = neighbor_seq.to(device)

            pred = model(center_seq, neighbor_seq)  # (bs, pred_steps, 3)
            all_preds[start:end] = pred.cpu().numpy()

    # Denormalize per variable
    var_keys = ["pressure", "u_velocity", "v_velocity"]
    var_labels = ["P", "U", "V"]
    for vi, (key, label, scaler) in enumerate(zip(var_keys, var_labels, scalers)):
        pred_v = all_preds[:, :, vi]  # (n_cells, pred_steps)
        pred_denorm = scaler.inverse_transform(pred_v.reshape(-1, 1)).reshape(n_cells, pred_steps)

        df_orig = dataframes_original[key]
        coord_cols = df_orig.iloc[:, :2]
        pred_cols = pd.DataFrame(
            pred_denorm,
            columns=[f"pred_{label}_t{i+1}" for i in range(pred_steps)],
        )
        df_out = pd.concat([coord_cols.reset_index(drop=True), pred_cols], axis=1)
        out_path = os.path.join(cfg["output_dir"], f"pred_{key}.csv")
        df_out.to_csv(out_path, index=False)
        print(f"  Saved {out_path}")

# Visualization

In [ ]:
def create_visualizations(predictions, targets, input_sequences, sample_indices, cfg):
    """
    Create time-series comparison plots (input + ground truth + prediction)
    for selected samples.  Saves one PNG per sample.

    Parameters
    ----------
    predictions    : np.ndarray, shape (N, pred_steps, 3)
    targets        : np.ndarray, shape (N, pred_steps, 3)
    input_sequences: np.ndarray, shape (N, seq_len, 3)  — normalized input history
    sample_indices : list of int — which samples to plot
    cfg            : CONFIG dict
    """
    var_names = ["Pressure (P)", "U-velocity", "V-velocity"]
    seq_len = input_sequences.shape[1] if input_sequences is not None else cfg["input_sequence_length"]
    pred_steps = predictions.shape[1]
    os.makedirs(cfg["output_dir"], exist_ok=True)

    for sample_idx in sample_indices:
        if sample_idx >= len(predictions):
            print(f"  Sample {sample_idx} out of range ({len(predictions)} available), skipping.")
            continue

        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        fig.suptitle(f"Sample {sample_idx} — Input + Prediction vs Ground Truth", fontsize=13)

        for vi, (ax, vname) in enumerate(zip(axes, var_names)):
            t_input = np.arange(seq_len)
            t_pred = np.arange(seq_len, seq_len + pred_steps)

            if input_sequences is not None:
                ax.plot(t_input, input_sequences[sample_idx, :, vi],
                        color="steelblue", label="Input (normalized)", linewidth=1.5)

            ax.plot(t_pred, targets[sample_idx, :, vi],
                    color="darkorange", label="Ground Truth", linewidth=2)
            ax.plot(t_pred, predictions[sample_idx, :, vi],
                    color="crimson", linestyle="--", label="Prediction", linewidth=2)

            ax.axvline(x=seq_len - 0.5, color="gray", linestyle=":", linewidth=1)
            ax.set_title(vname)
            ax.set_xlabel("Timestep")
            ax.set_ylabel("Normalized value")
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)

        plt.tight_layout()
        out_path = os.path.join(cfg["output_dir"], f"visualization_sample_{sample_idx}.png")
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"  Saved {out_path}")

# Main Pipeline

In [ ]:
def main():
    # ── 1. Load data ──────────────────────────────────────────────────────────
    coords, data, n_timesteps, dataframes_original = parse_urans_multifile(
        CONFIG["pressure_csv"],
        CONFIG["u_velocity_csv"],
        CONFIG["v_velocity_csv"],
        chunk_size=CONFIG["chunk_size"],
    )

    # ── 2. Build kNN graph ────────────────────────────────────────────────────
    neighbor_indices = build_knn_graph(coords, CONFIG["k_neighbors"])

    # ── 3. Temporal split boundaries ──────────────────────────────────────────
    train_end_t = int(CONFIG["train_ratio"] * n_timesteps)                # 70 %
    val_end_t   = int((CONFIG["train_ratio"] + CONFIG["val_ratio"]) * n_timesteps)  # 85 %
    print(f"Temporal split: train [0, {train_end_t}), "
          f"val [{train_end_t}, {val_end_t}), "
          f"test [{val_end_t}, {n_timesteps})")

    # ── 4. Normalize ──────────────────────────────────────────────────────────
    data_normalized, scalers = normalize_per_variable(data, train_end_t)

    # ── 5. Create datasets ────────────────────────────────────────────────────
    seq_len    = CONFIG["input_sequence_length"]
    pred_steps = CONFIG["prediction_horizon_steps"]

    train_dataset = SpatioTemporalDataset(
        data_normalized, neighbor_indices, seq_len, pred_steps, 0, train_end_t)
    val_dataset   = SpatioTemporalDataset(
        data_normalized, neighbor_indices, seq_len, pred_steps, train_end_t, val_end_t)
    test_dataset  = SpatioTemporalDataset(
        data_normalized, neighbor_indices, seq_len, pred_steps, val_end_t, n_timesteps)

    print(f"Dataset sizes — train: {len(train_dataset):,}, "
          f"val: {len(val_dataset):,}, test: {len(test_dataset):,}")

    # ── 6. DataLoaders ────────────────────────────────────────────────────────
    train_loader = DataLoader(
        train_dataset, batch_size=CONFIG["batch_size"],
        shuffle=True, num_workers=2, pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=CONFIG["batch_size"],
        shuffle=False, num_workers=2, pin_memory=True,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=CONFIG["batch_size"],
        shuffle=False, num_workers=2, pin_memory=True,
    )

    # ── 7. Model ──────────────────────────────────────────────────────────────
    model = create_model(CONFIG)
    if torch.cuda.device_count() > 1:
        print(f"Using DataParallel across {torch.cuda.device_count()} GPUs.")
        model = DataParallel(model)
    model = model.to(device)

    # ── 8. Optimizer, loss, scheduler ─────────────────────────────────────────
    optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
    criterion = WeightedMSELoss(CONFIG["loss_weights"]).to(device)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CONFIG["epochs"]
    )

    # ── 9. Training loop ──────────────────────────────────────────────────────
    best_val_loss = float("inf")
    patience_counter = 0
    best_model_path = os.path.join(CONFIG["output_dir"], "best_model.pth")
    history = {"train_loss": [], "val_loss": [],
               "pressure_rmse": [], "u_rmse": [], "v_rmse": []}

    print("\n" + "=" * 80)
    print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Val Loss':>10}  "
          f"{'Val RMSE':>9}  {'P_RMSE':>9}  {'U_RMSE':>9}  {'V_RMSE':>9}")
    print("=" * 80)

    for epoch in range(1, CONFIG["epochs"] + 1):
        train_metrics = train_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics   = evaluate(model, val_loader, criterion, scalers, device)
        scheduler.step()

        history["train_loss"].append(train_metrics["loss"])
        history["val_loss"].append(val_metrics["loss"])
        history["pressure_rmse"].append(val_metrics["pressure_rmse"])
        history["u_rmse"].append(val_metrics["u_rmse"])
        history["v_rmse"].append(val_metrics["v_rmse"])

        print(f"{epoch:>5}  {train_metrics['loss']:>10.6f}  {val_metrics['loss']:>10.6f}  "
              f"{val_metrics['rmse']:>9.5f}  {val_metrics['pressure_rmse']:>9.4f}  "
              f"{val_metrics['u_rmse']:>9.5f}  {val_metrics['v_rmse']:>9.5f}")

        # Save best model
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            patience_counter = 0
            # Unwrap DataParallel if needed
            state = model.module.state_dict() if isinstance(model, DataParallel) else model.state_dict()
            torch.save(state, best_model_path)
            print(f"         ↳ Best model saved (val_loss={best_val_loss:.6f})")
        else:
            patience_counter += 1
            if patience_counter >= CONFIG["patience"]:
                print(f"\nEarly stopping triggered after {epoch} epochs.")
                break

    # ── 10. Test evaluation ───────────────────────────────────────────────────
    print("\nLoading best model for test evaluation ...")
    raw_model = create_model(CONFIG).to(device)
    raw_model.load_state_dict(torch.load(best_model_path, map_location=device))
    if torch.cuda.device_count() > 1:
        raw_model = DataParallel(raw_model)

    test_metrics = evaluate(raw_model, test_loader, criterion, scalers, device)
    print("\nTest Results:")
    print(f"  Loss         : {test_metrics['loss']:.6f}")
    print(f"  RMSE         : {test_metrics['rmse']:.5f}")
    print(f"  MAE          : {test_metrics['mae']:.5f}")
    print(f"  Pressure RMSE: {test_metrics['pressure_rmse']:.4f}")
    print(f"  U-vel RMSE   : {test_metrics['u_rmse']:.5f}")
    print(f"  V-vel RMSE   : {test_metrics['v_rmse']:.5f}")

    # ── 11. Export predictions ────────────────────────────────────────────────
    if CONFIG["save_predictions"]:
        export_predictions_to_csv(
            raw_model, data_normalized, neighbor_indices,
            scalers, CONFIG, dataframes_original, device,
        )

    # ── 12. Visualizations ────────────────────────────────────────────────────
    if CONFIG["create_visualizations"]:
        # Collect a small batch of inputs from the test loader for plotting
        input_seqs_list = []
        for center_seq, _, _ in test_loader:
            input_seqs_list.append(center_seq.numpy())
            if len(np.concatenate(input_seqs_list, axis=0)) >= max(CONFIG["visualization_samples"]) + 1:
                break
        input_seqs_arr = np.concatenate(input_seqs_list, axis=0)

        create_visualizations(
            test_metrics["predictions"],
            test_metrics["targets"],
            input_seqs_arr,
            CONFIG["visualization_samples"],
            CONFIG,
        )

    # ── 13. Training history plot ─────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Training History", fontsize=13)

    epochs_range = range(1, len(history["train_loss"]) + 1)
    ax1.plot(epochs_range, history["train_loss"], label="Train Loss", color="steelblue")
    ax1.plot(epochs_range, history["val_loss"],   label="Val Loss",   color="darkorange")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Weighted MSE Loss")
    ax1.set_title("Loss Curves")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs_range, history["pressure_rmse"], label="Pressure RMSE", color="crimson")
    ax2.plot(epochs_range, history["u_rmse"],        label="U-vel RMSE",    color="seagreen")
    ax2.plot(epochs_range, history["v_rmse"],        label="V-vel RMSE",    color="mediumpurple")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("RMSE (physical units)")
    ax2.set_title("Per-Variable Validation RMSE")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    history_path = os.path.join(CONFIG["output_dir"], "training_history.png")
    plt.savefig(history_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Training history saved to {history_path}")

    return test_metrics


if __name__ == "__main__":
    main()